# NetworkX Comprehensive Training

This notebook provides a hands-on walkthrough of the **NetworkX** library — the standard Python package for creating, manipulating, and studying the structure, dynamics, and functions of complex networks.

## Table of Contents
1. [What is NetworkX?](#1)
2. [Graph Types](#2)
3. [Adding Nodes and Edges](#3)
4. [Node and Edge Attributes](#4)
5. [Visualizing Graphs](#5)
6. [Graph Properties and Inspection](#6)
7. [Graph Algorithms — Paths and Connectivity](#7)
8. [Centrality Measures](#8)
9. [Clustering and Communities](#9)
10. [Classic Graph Generators](#10)
11. [Working with Real-World Data](#11)
12. [Directed Graphs and Weighted Edges](#12)
13. [Saving and Loading Graphs](#13)

---
## 1. What is NetworkX? <a id='1'></a>

NetworkX is a Python package for the creation, manipulation, and study of the structure, dynamics, and functions of complex networks. It provides:

- Data structures for graphs, digraphs, and multigraphs
- Many standard graph algorithms
- Network structure and analysis measures
- Generators for classic graphs, random graphs, and synthetic networks
- Nodes can be *anything* — numbers, strings, images, custom objects
- Edges can carry arbitrary attribute data (weights, labels, etc.)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd

print(f"NetworkX version: {nx.__version__}")

---
## 2. Graph Types <a id='2'></a>

NetworkX provides four core graph classes:

| Class | Description |
|---|---|
| `Graph` | Undirected graph — edges have no direction |
| `DiGraph` | Directed graph — edges have direction (A→B ≠ B→A) |
| `MultiGraph` | Undirected multigraph — multiple edges allowed between same nodes |
| `MultiDiGraph` | Directed multigraph |


In [ ]:
# Undirected graph
G = nx.Graph()

# Directed graph
DG = nx.DiGraph()

# Multigraph (multiple edges between same pair of nodes)
MG = nx.MultiGraph()

print("Graph types created:")
print(f"  Undirected:  {type(G).__name__}")
print(f"  Directed:    {type(DG).__name__}")
print(f"  Multigraph:  {type(MG).__name__}")

---
## 3. Adding Nodes and Edges <a id='3'></a>

Nodes and edges can be added individually, in bulk from lists, or from other graphs.

In [ ]:
G = nx.Graph()

# --- Nodes ---
G.add_node(1)                        # single node
G.add_nodes_from([2, 3, 4, 5])       # list of nodes
G.add_nodes_from(range(6, 9))        # from a range

print(f"Nodes ({G.number_of_nodes()}): {list(G.nodes)}")

# --- Edges ---
G.add_edge(1, 2)                     # single edge
G.add_edges_from([(2, 3), (3, 4), (4, 5), (5, 1)])  # list of edges
G.add_edges_from([(1, 6), (2, 7), (3, 8)])           # more edges

print(f"Edges  ({G.number_of_edges()}): {list(G.edges)}")

In [ ]:
# Removing nodes and edges
G.remove_node(8)
G.remove_edge(2, 7)

print(f"After removals — nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")

# Nodes can be any hashable object
G_mixed = nx.Graph()
G_mixed.add_nodes_from(["Alice", "Bob", "Carol", 42, (0, 0)])
G_mixed.add_edges_from([("Alice", "Bob"), ("Bob", "Carol"), ("Carol", 42)])
print(f"Mixed node types: {list(G_mixed.nodes)}")

---
## 4. Node and Edge Attributes <a id='4'></a>

Any Python data can be stored as attributes on nodes and edges.

In [ ]:
# Build a social network with attributes
social = nx.Graph(name="Technodays Social Network")

# Add nodes with attributes
people = [
    ("Alice",  {"age": 30, "city": "Melbourne",  "role": "engineer"}),
    ("Bob",    {"age": 25, "city": "Sydney",     "role": "designer"}),
    ("Carol",  {"age": 35, "city": "Melbourne",  "role": "manager"}),
    ("Dave",   {"age": 28, "city": "Brisbane",   "role": "engineer"}),
    ("Eve",    {"age": 32, "city": "Sydney",     "role": "manager"}),
    ("Frank",  {"age": 27, "city": "Melbourne",  "role": "designer"}),
]
social.add_nodes_from(people)

# Add edges with attributes
friendships = [
    ("Alice", "Bob",   {"weight": 0.9, "since": 2018}),
    ("Alice", "Carol", {"weight": 0.7, "since": 2020}),
    ("Bob",   "Dave",  {"weight": 0.5, "since": 2021}),
    ("Carol", "Eve",   {"weight": 0.8, "since": 2019}),
    ("Dave",  "Eve",   {"weight": 0.6, "since": 2022}),
    ("Eve",   "Frank", {"weight": 0.4, "since": 2023}),
    ("Alice", "Frank", {"weight": 0.3, "since": 2023}),
]
social.add_edges_from(friendships)

# Access attributes
print("Alice's attributes:", social.nodes["Alice"])
print("Alice-Bob edge:    ", social.edges["Alice", "Bob"])

# Update an attribute
social.nodes["Alice"]["city"] = "Canberra"
print("Alice's updated city:", social.nodes["Alice"]["city"])

In [ ]:
# Iterate over node attributes
print("People and their roles:")
for name, attrs in social.nodes(data=True):
    print(f"  {name:8s} — {attrs['role']:10s} in {attrs['city']}")

print("\nFriendships and weights:")
for u, v, attrs in social.edges(data=True):
    print(f"  {u} ↔ {v}  weight={attrs['weight']:.1f}  since={attrs['since']}")

---
## 5. Visualizing Graphs <a id='5'></a>

NetworkX integrates with Matplotlib for drawing. Layout algorithms determine node positions.

In [ ]:
# Basic drawing
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Spring layout (force-directed) — default
pos_spring = nx.spring_layout(social, seed=42)
nx.draw(social, pos_spring, ax=axes[0],
        with_labels=True, node_color="steelblue",
        node_size=1000, font_color="white", font_weight="bold",
        edge_color="gray", width=2)
axes[0].set_title("Spring Layout", fontsize=14)

# Circular layout
pos_circ = nx.circular_layout(social)
nx.draw(social, pos_circ, ax=axes[1],
        with_labels=True, node_color="coral",
        node_size=1000, font_color="white", font_weight="bold",
        edge_color="gray", width=2)
axes[1].set_title("Circular Layout", fontsize=14)

plt.suptitle("Social Network — Layout Comparison", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Styled drawing: colour nodes by role, scale edges by weight
role_colours = {"engineer": "#4C72B0", "designer": "#DD8452", "manager": "#55A868"}
node_colors = [role_colours[social.nodes[n]["role"]] for n in social.nodes]
edge_widths  = [social.edges[e]["weight"] * 5 for e in social.edges]
edge_labels  = {(u, v): f"{d['weight']:.1f}" for u, v, d in social.edges(data=True)}

pos = nx.spring_layout(social, seed=42)
fig, ax = plt.subplots(figsize=(10, 7))

nx.draw_networkx_nodes(social, pos, node_color=node_colors, node_size=1200, ax=ax)
nx.draw_networkx_labels(social, pos, font_color="white", font_weight="bold", font_size=11, ax=ax)
nx.draw_networkx_edges(social, pos, width=edge_widths, edge_color="#888888", ax=ax)
nx.draw_networkx_edge_labels(social, pos, edge_labels=edge_labels, font_size=9, ax=ax)

# Legend
from matplotlib.patches import Patch
legend = [Patch(color=c, label=r) for r, c in role_colours.items()]
ax.legend(handles=legend, loc="upper left", title="Role")

ax.set_title("Social Network — Styled Visualisation", fontsize=15)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Available layout algorithms
layouts = {
    "Spring":     nx.spring_layout(social, seed=42),
    "Circular":   nx.circular_layout(social),
    "Kamada-Kawai": nx.kamada_kawai_layout(social),
    "Spectral":   nx.spectral_layout(social),
    "Random":     nx.random_layout(social, seed=42),
    "Shell":      nx.shell_layout(social),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for ax, (name, pos) in zip(axes, layouts.items()):
    nx.draw(social, pos, ax=ax, with_labels=True,
            node_color="mediumpurple", node_size=700,
            font_color="white", font_size=9, edge_color="gray")
    ax.set_title(name, fontsize=12)

plt.suptitle("Layout Algorithm Comparison", fontsize=16)
plt.tight_layout()
plt.show()

---
## 6. Graph Properties and Inspection <a id='6'></a>

In [ ]:
G = social  # reuse our social network

print("=== Basic Properties ===")
print(f"Nodes:           {G.number_of_nodes()}")
print(f"Edges:           {G.number_of_edges()}")
print(f"Is directed:     {G.is_directed()}")
print(f"Is weighted:     {nx.is_weighted(G)}")
print(f"Is connected:    {nx.is_connected(G)}")

print("\n=== Degree (number of connections per node) ===")
for node, degree in sorted(G.degree(), key=lambda x: -x[1]):
    neighbours = list(G.neighbors(node))
    print(f"  {node:8s}  degree={degree}  neighbours={neighbours}")

print(f"\nAverage degree:  {sum(d for _, d in G.degree()) / G.number_of_nodes():.2f}")
print(f"Density:         {nx.density(G):.4f}")

In [ ]:
# Degree distribution histogram
degrees = [d for _, d in G.degree()]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(*np.unique(degrees, return_counts=True), color="steelblue", edgecolor="white")
ax.set_xlabel("Degree", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Degree Distribution", fontsize=14)
ax.set_xticks(range(min(degrees), max(degrees) + 1))
plt.tight_layout()
plt.show()

---
## 7. Graph Algorithms — Paths and Connectivity <a id='7'></a>

In [ ]:
# Shortest paths
path = nx.shortest_path(social, source="Alice", target="Eve")
length = nx.shortest_path_length(social, source="Alice", target="Eve")
print(f"Shortest path Alice → Eve:  {' → '.join(path)}  (length {length})")

# Weighted shortest path (uses 1/weight so higher weight = shorter distance)
# First set 'distance' as inverse of weight
for u, v, d in social.edges(data=True):
    social[u][v]["distance"] = 1 - d["weight"]  # lower weight = longer path

w_path = nx.shortest_path(social, source="Alice", target="Eve", weight="distance")
print(f"Weighted shortest path:      {' → '.join(w_path)}")

# All-pairs shortest paths
print("\nAll-pairs path lengths:")
apsp = dict(nx.all_pairs_shortest_path_length(social))
nodes = list(social.nodes)
# Display as a matrix
df = pd.DataFrame([[apsp[u][v] for v in nodes] for u in nodes], index=nodes, columns=nodes)
print(df.to_string())

In [ ]:
# Graph diameter and radius
print(f"Diameter (longest shortest path): {nx.diameter(social)}")
print(f"Radius:                           {nx.radius(social)}")
print(f"Center nodes:                     {nx.center(social)}")
print(f"Periphery nodes:                  {nx.periphery(social)}")

# Connected components
# Add an isolated subgraph to demonstrate components
G_comp = social.copy()
G_comp.add_edges_from([("X", "Y"), ("Y", "Z")])

components = list(nx.connected_components(G_comp))
print(f"\nConnected components in extended graph: {len(components)}")
for i, comp in enumerate(components):
    print(f"  Component {i+1}: {comp}")

# Find bridges (edges whose removal disconnects the graph)
bridges = list(nx.bridges(social))
print(f"\nBridge edges (critical connections): {bridges}")

In [ ]:
# Visualise shortest path
path = nx.shortest_path(social, source="Alice", target="Eve")
path_edges = list(zip(path, path[1:]))

pos = nx.spring_layout(social, seed=42)
fig, ax = plt.subplots(figsize=(9, 6))

# Draw all nodes and edges
nx.draw_networkx_nodes(social, pos, node_color="lightgray", node_size=900, ax=ax)
nx.draw_networkx_edges(social, pos, edge_color="lightgray", width=1, ax=ax)

# Highlight path
nx.draw_networkx_nodes(social, pos, nodelist=path,
                       node_color="tomato", node_size=1100, ax=ax)
nx.draw_networkx_edges(social, pos, edgelist=path_edges,
                       edge_color="tomato", width=4, ax=ax)
nx.draw_networkx_labels(social, pos, font_weight="bold", ax=ax)

ax.set_title(f"Shortest Path: {' → '.join(path)}", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

---
## 8. Centrality Measures <a id='8'></a>

Centrality identifies the most important nodes in a network.

| Measure | What it captures |
|---|---|
| **Degree centrality** | Number of direct connections |
| **Betweenness centrality** | How often a node lies on shortest paths |
| **Closeness centrality** | How quickly a node can reach all others |
| **Eigenvector centrality** | Connection to other well-connected nodes |
| **PageRank** | Recursive importance (like Google's algorithm) |

In [ ]:
centrality_measures = {
    "Degree":       nx.degree_centrality(social),
    "Betweenness":  nx.betweenness_centrality(social),
    "Closeness":    nx.closeness_centrality(social),
    "Eigenvector":  nx.eigenvector_centrality(social, max_iter=1000),
    "PageRank":     nx.pagerank(social),
}

df_cent = pd.DataFrame(centrality_measures).round(3)
df_cent.index.name = "Node"
print(df_cent.to_string())

print("\nMost influential node per measure:")
for measure, values in centrality_measures.items():
    top = max(values, key=values.get)
    print(f"  {measure:15s}: {top}  ({values[top]:.3f})")

In [ ]:
# Visualise betweenness centrality as node size
bc = nx.betweenness_centrality(social)
node_sizes  = [5000 * bc[n] + 400 for n in social.nodes]
node_colors = [bc[n] for n in social.nodes]

pos = nx.spring_layout(social, seed=42)
fig, ax = plt.subplots(figsize=(9, 6))

nodes = nx.draw_networkx_nodes(social, pos, node_size=node_sizes,
                                node_color=node_colors, cmap=cm.plasma, ax=ax)
nx.draw_networkx_edges(social, pos, edge_color="gray", alpha=0.6, ax=ax)
nx.draw_networkx_labels(social, pos, font_color="white", font_weight="bold", ax=ax)

plt.colorbar(nodes, ax=ax, label="Betweenness Centrality")
ax.set_title("Node Size & Colour ∝ Betweenness Centrality", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

---
## 9. Clustering and Communities <a id='9'></a>

Clustering measures how tightly a node's neighbours are interconnected. Community detection finds dense subgraphs.

In [ ]:
# Local clustering coefficient
clust = nx.clustering(social)
print("Local clustering coefficients:")
for node, coef in sorted(clust.items(), key=lambda x: -x[1]):
    print(f"  {node:8s}: {coef:.3f}")

print(f"\nGlobal clustering coefficient (transitivity): {nx.transitivity(social):.3f}")
print(f"Average clustering coefficient:               {nx.average_clustering(social):.3f}")

In [ ]:
# Community detection — Girvan-Newman algorithm
# Build a larger, community-structured graph for a clearer demo
community_graph = nx.karate_club_graph()  # classic community benchmark graph

from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(community_graph))
print(f"Karate Club graph: {community_graph.number_of_nodes()} nodes, "
      f"{community_graph.number_of_edges()} edges")
print(f"Communities found: {len(communities)}")
for i, c in enumerate(communities):
    print(f"  Community {i+1} ({len(c)} members): {sorted(c)}")

In [ ]:
# Visualise communities
colour_map = {}
palette = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]
for i, community in enumerate(communities):
    for node in community:
        colour_map[node] = palette[i % len(palette)]

node_colors = [colour_map[n] for n in community_graph.nodes]
pos = nx.spring_layout(community_graph, seed=42)

fig, ax = plt.subplots(figsize=(11, 8))
nx.draw(community_graph, pos, ax=ax,
        node_color=node_colors, node_size=300,
        with_labels=True, font_size=8, edge_color="gray", alpha=0.85)

from matplotlib.patches import Patch
legend = [Patch(color=palette[i], label=f"Community {i+1}")
          for i in range(len(communities))]
ax.legend(handles=legend, loc="upper left")
ax.set_title("Karate Club Graph — Community Detection", fontsize=14)
plt.tight_layout()
plt.show()

---
## 10. Classic Graph Generators <a id='10'></a>

NetworkX ships with dozens of generators for well-known and random graph families.

In [ ]:
generators = {
    "Complete K5":         nx.complete_graph(5),
    "Petersen":            nx.petersen_graph(),
    "Barbell (3-3)": nx.barbell_graph(3, 2),
    "Erdős–Rényi":         nx.erdos_renyi_graph(15, 0.3, seed=42),
    "Barabási–Albert":     nx.barabasi_albert_graph(20, 2, seed=42),
    "Watts–Strogatz":      nx.watts_strogatz_graph(15, 4, 0.3, seed=42),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
colours = ["steelblue", "coral", "mediumseagreen", "mediumpurple", "goldenrod", "tomato"]

for ax, (name, g), colour in zip(axes, generators.items(), colours):
    pos = nx.spring_layout(g, seed=42)
    nx.draw(g, pos, ax=ax, with_labels=True,
            node_color=colour, node_size=400,
            font_color="white", font_size=8, edge_color="gray")
    ax.set_title(f"{name}\n({g.number_of_nodes()}N, {g.number_of_edges()}E)", fontsize=11)

plt.suptitle("Classic and Random Graph Generators", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Spanning trees
G_rand = nx.barabasi_albert_graph(12, 2, seed=42)
mst = nx.minimum_spanning_tree(G_rand)

pos = nx.spring_layout(G_rand, seed=42)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

nx.draw(G_rand, pos, ax=axes[0], with_labels=True,
        node_color="steelblue", node_size=500, font_color="white", edge_color="gray")
axes[0].set_title("Original Graph", fontsize=13)

nx.draw(mst, pos, ax=axes[1], with_labels=True,
        node_color="steelblue", node_size=500, font_color="white", edge_color="tomato", width=2)
axes[1].set_title("Minimum Spanning Tree", fontsize=13)

plt.tight_layout()
plt.show()

---
## 11. Working with Real-World Data <a id='11'></a>

Here we build a graph from a pandas DataFrame — a common real-world workflow.

In [ ]:
# Simulate a flight route dataset
np.random.seed(42)
cities = ["Melbourne", "Sydney", "Brisbane", "Perth", "Adelaide",
          "Hobart", "Darwin", "Canberra"]

routes = [
    ("Melbourne", "Sydney",    1.1),
    ("Melbourne", "Adelaide",  1.3),
    ("Melbourne", "Canberra",  0.9),
    ("Melbourne", "Hobart",    1.4),
    ("Sydney",    "Brisbane",  1.5),
    ("Sydney",    "Canberra",  0.6),
    ("Sydney",    "Perth",     4.5),
    ("Brisbane",  "Darwin",    4.0),
    ("Adelaide",  "Perth",     2.7),
    ("Adelaide",  "Darwin",    3.8),
    ("Perth",     "Darwin",    3.3),
]

df_routes = pd.DataFrame(routes, columns=["from", "to", "hours"])
print(df_routes.to_string(index=False))

# Build weighted graph from DataFrame
flights = nx.from_pandas_edgelist(
    df_routes, source="from", target="to", edge_attr="hours"
)

print(f"\nFlight network: {flights.number_of_nodes()} cities, {flights.number_of_edges()} routes")

In [ ]:
# Fastest route between two cities
origin, destination = "Hobart", "Darwin"
fast_path = nx.shortest_path(flights, origin, destination, weight="hours")
total_hours = sum(flights[fast_path[i]][fast_path[i+1]]["hours"]
                  for i in range(len(fast_path) - 1))
print(f"Fastest route {origin} → {destination}: {' → '.join(fast_path)}")
print(f"Total flight time: {total_hours:.1f} hours")

# Visualise flight map
# Approximate lat/lon positions for layout
geo_pos = {
    "Melbourne": (144.9, -37.8),  "Sydney":    (151.2, -33.9),
    "Brisbane":  (153.0, -27.5),  "Perth":     (115.9, -32.0),
    "Adelaide":  (138.6, -34.9),  "Hobart":    (147.3, -42.9),
    "Darwin":    (130.8, -12.5),  "Canberra":  (149.1, -35.3),
}

edge_labels = {(u, v): f"{d:.1f}h" for u, v, d in flights.edges(data="hours")}
path_edges  = list(zip(fast_path, fast_path[1:]))

fig, ax = plt.subplots(figsize=(11, 8))
nx.draw_networkx_nodes(flights, geo_pos, node_size=800,
                       node_color="#4C72B0", ax=ax)
nx.draw_networkx_edges(flights, geo_pos, edge_color="lightgray", width=1.5, ax=ax)
nx.draw_networkx_edges(flights, geo_pos, edgelist=path_edges,
                       edge_color="tomato", width=3.5, ax=ax)
nx.draw_networkx_labels(flights, geo_pos, font_color="white",
                        font_weight="bold", font_size=9, ax=ax)
nx.draw_networkx_edge_labels(flights, geo_pos, edge_labels=edge_labels,
                              font_size=8, ax=ax)

ax.set_title(f"Australian Flight Network — Fastest Route: {' → '.join(fast_path)}",
             fontsize=13)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

---
## 12. Directed Graphs and Weighted Edges <a id='12'></a>

Directed graphs (DiGraphs) model asymmetric relationships — web links, dependencies, information flow.

In [ ]:
# Model a task dependency graph
tasks = nx.DiGraph()
tasks.add_nodes_from([
    ("Design",      {"duration": 3}),
    ("Backend",     {"duration": 5}),
    ("Frontend",    {"duration": 4}),
    ("Database",    {"duration": 2}),
    ("Testing",     {"duration": 3}),
    ("Deployment",  {"duration": 1}),
])

# Edge = 'must complete before'
tasks.add_edges_from([
    ("Design",   "Backend"),
    ("Design",   "Frontend"),
    ("Design",   "Database"),
    ("Backend",  "Testing"),
    ("Frontend", "Testing"),
    ("Database", "Backend"),
    ("Testing",  "Deployment"),
])

print("Is a DAG (no cycles):", nx.is_directed_acyclic_graph(tasks))
print("Topological order:  ", list(nx.topological_sort(tasks)))

# In-degree and out-degree
print("\nNode degrees:")
for n in tasks.nodes:
    print(f"  {n:12s}  in={tasks.in_degree(n)}  out={tasks.out_degree(n)}")

In [ ]:
# Visualise DAG
pos = nx.nx_agraph.graphviz_layout(tasks, prog="dot") if False else nx.spring_layout(tasks, seed=7)
# Use a simple layered layout by topological level
topo = list(nx.topological_sort(tasks))
layer = {n: i for i, n in enumerate(topo)}
pos = {n: (layer[n] * 2, np.random.uniform(-0.5, 0.5)) for n in tasks.nodes}

# Reposition nicely
pos = {
    "Design":     (0, 0),
    "Database":   (2, 1),
    "Frontend":   (2, -1),
    "Backend":    (4, 1),
    "Testing":    (6, 0),
    "Deployment": (8, 0),
}

durations = {n: d["duration"] for n, d in tasks.nodes(data=True)}
node_sizes = [durations[n] * 300 for n in tasks.nodes]
node_labels = {n: f"{n}\n({durations[n]}d)" for n in tasks.nodes}

fig, ax = plt.subplots(figsize=(13, 5))
nx.draw_networkx_nodes(tasks, pos, node_size=node_sizes,
                       node_color="#4C72B0", ax=ax)
nx.draw_networkx_labels(tasks, pos, labels=node_labels,
                        font_color="white", font_size=9, ax=ax)
nx.draw_networkx_edges(tasks, pos, arrowsize=20, arrowstyle="->",
                       edge_color="gray", width=2,
                       connectionstyle="arc3,rad=0.1", ax=ax)
ax.set_title("Project Task Dependency Graph (DAG)", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# PageRank on a directed web-link graph
web = nx.DiGraph()
pages = ["Home", "About", "Products", "Blog", "Contact", "Docs"]
links = [
    ("Home", "Products"), ("Home", "Blog"),    ("Home", "About"),
    ("Blog", "Home"),     ("Blog", "Docs"),    ("Blog", "Products"),
    ("Products", "Home"),("Products", "Docs"),("Docs", "Products"),
    ("About", "Home"),   ("About", "Contact"),("Contact", "Home"),
]
web.add_nodes_from(pages)
web.add_edges_from(links)

pr = nx.pagerank(web, alpha=0.85)
print("PageRank scores (web graph):")
for page, score in sorted(pr.items(), key=lambda x: -x[1]):
    bar = "█" * int(score * 100)
    print(f"  {page:10s}: {score:.4f}  {bar}")

---
## 13. Saving and Loading Graphs <a id='13'></a>

NetworkX supports multiple serialisation formats.

In [ ]:
import json
from pathlib import Path

G_save = social

# --- GraphML (XML, widely supported) ---
nx.write_graphml(G_save, "social_network.graphml")
G_loaded = nx.read_graphml("social_network.graphml")
print(f"GraphML round-trip: {G_loaded.number_of_nodes()} nodes, {G_loaded.number_of_edges()} edges")

# --- JSON node-link format ---
data = nx.node_link_data(G_save)
with open("social_network.json", "w") as f:
    json.dump(data, f, indent=2)

with open("social_network.json") as f:
    data_loaded = json.load(f)
G_from_json = nx.node_link_graph(data_loaded)
print(f"JSON round-trip:    {G_from_json.number_of_nodes()} nodes, {G_from_json.number_of_edges()} edges")

# --- Adjacency matrix ---
adj = nx.to_numpy_array(social)
print(f"\nAdjacency matrix shape: {adj.shape}")
df_adj = pd.DataFrame(adj, index=list(social.nodes), columns=list(social.nodes))
print(df_adj.to_string())

# Rebuild from adjacency matrix
G_from_adj = nx.from_numpy_array(adj)
print(f"\nRebuilt from matrix: {G_from_adj.number_of_nodes()} nodes, {G_from_adj.number_of_edges()} edges")

# Clean up
for f in ["social_network.graphml", "social_network.json"]:
    Path(f).unlink(missing_ok=True)
print("Temp files cleaned up.")

---
## Summary

You now know how to:

| Topic | Key functions |
|---|---|
| Create graphs | `nx.Graph()`, `nx.DiGraph()` |
| Add nodes/edges | `.add_node()`, `.add_edges_from()` |
| Node/edge attributes | `.nodes[n]`, `.edges[u,v]` |
| Visualise | `nx.draw()`, `nx.draw_networkx_*()` |
| Layouts | `spring_layout`, `circular_layout`, `kamada_kawai_layout` |
| Paths | `nx.shortest_path()`, `nx.all_pairs_shortest_path_length()` |
| Centrality | `degree_centrality`, `betweenness_centrality`, `pagerank` |
| Community | `greedy_modularity_communities` |
| Generators | `complete_graph`, `barabasi_albert_graph`, `watts_strogatz_graph` |
| Real data | `nx.from_pandas_edgelist()` |
| Directed graphs | `DiGraph`, `topological_sort`, `is_directed_acyclic_graph` |
| Serialisation | `write_graphml`, `node_link_data`, `to_numpy_array` |

### Next Steps
- Explore `networkx.algorithms` — there are 200+ algorithms available
- Try `nx.draw_networkx()` with custom `matplotlib` figures for publication-quality plots
- For very large graphs, consider **graph-tool** or **igraph** for C-backed performance
- Use **Gephi** (import GraphML files) for interactive visual exploration